# 10 — Bulk combined analysis: effect-size meta-analysis + leave-one-cohort-out incremental test

Question (posed Nov 23, 2026): pooled across the four bulk cohorts, how strong is the
IAF program's association with anti-TNF non-response — and does SIG9 add anything
ON TOP of the known comparators (OSM + generic inflammation) out of cohort?

Design: five independent strata (GSE12251 UC; GSE16879 UC; GSE16879 CD; GSE23597 UC;
GSE73661 UC-IFX). Scores computed exactly as in notebook 09 (per-stratum gene
z-scores; no cross-cohort leakage). (1) Hedges g per stratum, DerSimonian-Laird
random-effects pooling per score. (2) Leave-one-cohort-out logistic regression:
model A = OSM + INFLAM6; model B = A + SIG9; model C = SIG9 alone; delta = B - A.
VM-verified Nov 23, 2026; this notebook reproduces it.


In [ ]:
import gzip, io, json, math, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA = Path.home() / "Documents/immune-endotypes/data/bulk_response"
DATA.mkdir(parents=True, exist_ok=True)
MOD = Path.home() / "Documents/github/immune-endotypes-portfolio/modules/05_cross_cohort"
PROBES = json.loads((MOD / "sig_probes.json").read_text())
SIG9  = ["IL11","IL13RA2","IL24","CHI3L1","MMP3","MMP10","IL6","CXCL1","CXCL5"]
SIG25 = ["MMP1","SAA1","IL24","MASP1","WFDC2","C15orf48","CD82","TWIST2","SDC4","COL8A1",
         "SLC16A3","MME","CXCL5","CEMIP","CSF3","PTGS1","MMP3","SAA2","GK","TMEM132A",
         "CDCP1","ACKR4","SERINC2","AKR1B1","SQRDL"]
INFLAM= ["S100A8","S100A9","IL1B","CXCL8","LCN2","REG1A"]
URLS = {
 "GSE12251": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE12nnn/GSE12251/matrix/GSE12251_series_matrix.txt.gz",
 "GSE16879": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz",
 "GSE73661": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE73nnn/GSE73661/matrix/GSE73661_series_matrix.txt.gz",
 "GSE23597": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE23nnn/GSE23597/matrix/GSE23597_series_matrix.txt.gz",
}
for gse, url in URLS.items():
    out = DATA / f"{gse}_series_matrix.txt.gz"
    if not out.exists(): subprocess.run(["curl","-sSL","-o",str(out),url], check=True)
print("data ready")


In [ ]:
def load_series(path):
    titles=[]; char=[]; tbl=[]; in_tbl=False
    with gzip.open(path,"rt",errors="replace") as f:
        for line in f:
            if line.startswith("!series_matrix_table_begin"): in_tbl=True; continue
            if line.startswith("!series_matrix_table_end"): break
            if in_tbl: tbl.append(line); continue
            if line.startswith("!Sample_title"): titles=[v.strip('"') for v in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics"): char.append([v.strip('"') for v in line.rstrip("\n").split("\t")[1:]])
    df=pd.read_csv(io.StringIO("".join(tbl)),sep="\t",index_col=0)
    df.columns=[c.strip('"') for c in df.columns]; df.index=df.index.astype(str)
    return df,titles,char
def cs(char,key):
    for row in char:
        if row[0].split(":")[0].strip().lower()==key:
            return pd.Series([r.split(":",1)[1].strip() if ":" in r else r for r in row])
def gm_of(df,pmap):
    inv={g:[p for p in ps if p in df.index] for g,ps in pmap.items()}; inv={g:ps for g,ps in inv.items() if ps}
    return pd.DataFrame({g: df.loc[ps].mean(axis=0) for g,ps in inv.items()}).T
def scores(gm, labels):
    keep=labels.dropna(); g=gm[keep.index]
    z=g.sub(g.mean(axis=1),axis=0).div(g.std(axis=1)+1e-9,axis=0)
    def sc(genes):
        gs=[x for x in genes if x in z.index]
        return (z.loc[gs].mean(axis=0) if len(gs)>1 else z.loc[gs[0]]).values
    return pd.DataFrame({"SIG25":sc(SIG25),"SIG9":sc(SIG9),"OSM":sc(["OSM"]),"INFLAM6":sc(INFLAM),
                         "y":(keep.values=="NR").astype(int)}, index=keep.index)

strata={}
df,t,ch=load_series(DATA/"GSE12251_series_matrix.txt.gz"); resp=cs(ch,"wk8rsphm")
strata["GSE12251_UC"]=scores(gm_of(df,PROBES["GPL570"]), pd.Series({df.columns[i]:("R" if resp[i]=="Yes" else "NR") for i in range(len(t))}))
df,t,ch=load_series(DATA/"GSE16879_series_matrix.txt.gz")
dis=cs(ch,"disease"); tim=cs(ch,"before or after first infliximab treatment"); resp=cs(ch,"response to infliximab")
g=gm_of(df,PROBES["GPL570"])
for strat in ["UC","CD"]:
    lab={df.columns[i]:("R" if resp[i]=="Yes" else "NR") for i in range(len(t)) if tim[i].startswith("Before") and resp[i] in ("Yes","No") and dis[i]==strat}
    strata[f"GSE16879_{strat}"]=scores(g, pd.Series(lab))
df,t,ch=load_series(DATA/"GSE23597_series_matrix.txt.gz")
tim=cs(ch,"time"); dose=cs(ch,"dose"); resp=cs(ch,"wk8 response")
strata["GSE23597_UC"]=scores(gm_of(df,PROBES["GPL570"]), pd.Series({df.columns[i]:("R" if resp[i]=="Yes" else "NR") for i in range(len(t)) if tim[i]=="W0" and dose[i]!="placebo"}))
df,t,ch=load_series(DATA/"GSE73661_series_matrix.txt.gz")
week=cs(ch,"week (w)"); ther=cs(ch,"induction therapy_maintenance therapy"); indiv=cs(ch,"study individual number"); mayo=cs(ch,"mayo endoscopic subscore")
mmap={indiv[i]:mayo[i] for i in range(len(t)) if week[i]=="W4_W6" and ther[i]=="IFX"}
lab={df.columns[i]:("R" if mmap[indiv[i]] in ("0","1") else "NR") for i in range(len(t)) if week[i]=="W0" and ther[i]=="IFX" and indiv[i] in mmap}
strata["GSE73661_UC"]=scores(gm_of(df,PROBES["GPL6244"]), pd.Series(lab))
for k,v in strata.items(): print(k, "n=", len(v), "NR=", int(v.y.sum()))


In [ ]:
def hedges(x_nr, x_r):
    n1,n2=len(x_nr),len(x_r)
    sp=math.sqrt(((n1-1)*x_nr.var(ddof=1)+(n2-1)*x_r.var(ddof=1))/(n1+n2-2))
    d=(x_nr.mean()-x_r.mean())/sp; J=1-3/(4*(n1+n2)-9); g=J*d
    se=math.sqrt((n1+n2)/(n1*n2) + g*g/(2*(n1+n2-2)))
    return g,se
def dl_meta(gs, ses):
    gs=np.array(gs); ses=np.array(ses); k=len(gs); w=1/ses**2
    mu_fe=(w*gs).sum()/w.sum(); Q=float((w*(gs-mu_fe)**2).sum())
    tau2=max(0.0,(Q-(k-1))/(w.sum()-(w**2).sum()/w.sum()))
    wr=1/(ses**2+tau2); mu=(wr*gs).sum()/wr.sum(); se=math.sqrt(1/wr.sum())
    I2=max(0.0,(Q-(k-1))/Q)*100 if Q>0 else 0.0
    return mu,se,I2
meta_rows=[]
for score in ["SIG25","SIG9","OSM","INFLAM6"]:
    gs=[]; ses=[]
    for name,S in strata.items():
        gg,ss=hedges(S.loc[S.y==1,score].values, S.loc[S.y==0,score].values); gs.append(gg); ses.append(ss)
    mu,se,I2=dl_meta(gs,ses); p=2*(1-stats.norm.cdf(abs(mu/se)))
    meta_rows.append({"score":score,"pooled_g":round(mu,2),"CI_lo":round(mu-1.96*se,2),"CI_hi":round(mu+1.96*se,2),"p":float(f"{p:.1e}"),"I2_pct":round(I2)})
    print(meta_rows[-1])
pd.DataFrame(meta_rows).to_csv(MOD/"results/bulk_combined_meta.csv", index=False)


In [ ]:
def fit_logreg(X,y,lam=1.0,iters=3000,lr=0.2):
    mu,sd=X.mean(0),X.std(0)+1e-9; Xs=(X-mu)/sd
    Xb=np.c_[np.ones(len(Xs)),Xs]; w=np.zeros(Xb.shape[1])
    for _ in range(iters):
        p=1/(1+np.exp(-Xb@w)); w-=lr*(Xb.T@(p-y)/len(y)+lam*np.r_[0,w[1:]]/len(y))
    return w,mu,sd
def pred_logreg(model,X):
    w,mu,sd=model; Xs=(X-mu)/sd; Xb=np.c_[np.ones(len(Xs)),Xs]
    return 1/(1+np.exp(-Xb@w))
def auc(y,p):
    U,_=stats.mannwhitneyu(p[y==1],p[y==0],alternative="two-sided")
    return float(U/((y==1).sum()*(y==0).sum()))
lodo_rows=[]; names=list(strata)
for held in names:
    train=pd.concat([strata[n] for n in names if n!=held]); test=strata[held]
    ytr,yte=train.y.values,test.y.values
    aA=auc(yte, pred_logreg(fit_logreg(train[["OSM","INFLAM6"]].values,ytr), test[["OSM","INFLAM6"]].values))
    aB=auc(yte, pred_logreg(fit_logreg(train[["OSM","INFLAM6","SIG9"]].values,ytr), test[["OSM","INFLAM6","SIG9"]].values))
    aC=auc(yte, pred_logreg(fit_logreg(train[["SIG9"]].values,ytr), test[["SIG9"]].values))
    lodo_rows.append({"held_out":held,"AUC_A_OSM_INFLAM":round(aA,3),"AUC_B_plusSIG9":round(aB,3),
                      "delta_B_minus_A":round(aB-aA,3),"AUC_C_SIG9only":round(aC,3)})
    print(lodo_rows[-1])
print("mean delta:", round(float(np.mean([r["delta_B_minus_A"] for r in lodo_rows])),3))
pd.DataFrame(lodo_rows).to_csv(MOD/"results/bulk_combined_lodo.csv", index=False)


## Read-out (VM-verified Nov 23, 2026)

Meta-analysis: SIG9 has the LARGEST pooled effect of the four scores
(g = 1.44 [1.05, 1.84], p = 8.6e-13), ahead of SIG25 (1.18), INFLAM6 (1.13) and
OSM (0.96), with I-squared = 0% for every score — the association is unusually
consistent across cohorts.

LODO incremental test: adding SIG9 to OSM + INFLAM6 gives delta AUC of
-0.023 / +0.094 / -0.068 / +0.057 / +0.125 (mean +0.037) — positive in 3 of 5
held-out cohorts, negative in 2 (GSE12251, where OSM alone is near-perfect, and
GSE16879 CD, where generic inflammation dominates). SIG9 alone is competitive
with the two-marker comparator model in several cohorts (0.945 and 0.867 held-out
AUCs). Verdict, stated plainly: SIG9 is among the strongest and most consistent
single scores, but it does NOT add a consistent out-of-cohort gain on top of
OSM + inflammation. The bulk evidence stays replication-grade; the program's
distinctiveness rests on its single-cell identity and the severity-matched
stratum result (notebook 09), not on superior bulk prediction.
